## Imports & Pfade

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

from scipy.signal import butter, filtfilt, welch

DATA_ROOT = Path("../data")
HARM_DIR = DATA_ROOT / "processed" / "harmonized_250hz"
SEG_PATH = DATA_ROOT / "processed" / "segments_5s_2p5s_index_clean.parquet" 
OUT_DIR  = DATA_ROOT / "processed" / "features"
OUT_DIR.mkdir(parents=True, exist_ok=True)

assert HARM_DIR.exists(), f"Nicht gefunden: {HARM_DIR.resolve()}"
print("HARM_DIR:", HARM_DIR.resolve())



## Segmentindex laden

In [ ]:
seg_index = pd.read_parquet(SEG_PATH)
print("Segments loaded:", len(seg_index))
print(seg_index["label"].value_counts())
seg_index.head()


## Settings

In [ ]:
FS = 250
BANDPASS_ON = True
BP_LOW = 0.5
BP_HIGH = 40.0
BP_ORDER = 4

N_PER_SEG = 256 


## Bandpass Filter (0,5-40 Hz)

In [ ]:
def bandpass_filter(x: np.ndarray, fs: int, low: float = 0.5, high: float = 40.0, order: int = 4) -> np.ndarray:
    nyq = 0.5 * fs
    low_n = low / nyq
    high_n = high / nyq
    b, a = butter(order, [low_n, high_n], btype="bandpass")
    return filtfilt(b, a, x)


## Feature-Funktionen (pro 1D Kanal)

In [ ]:
def zero_crossing_rate(x: np.ndarray) -> float:
    s = np.sign(x)
    return float(np.mean(s[1:] * s[:-1] < 0))

def rms(x: np.ndarray) -> float:
    return float(np.sqrt(np.mean(x**2)))

def bandpower(f: np.ndarray, Pxx: np.ndarray, fmin: float, fmax: float) -> float:
    mask = (f >= fmin) & (f <= fmax)
    if not np.any(mask):
        return 0.0
    return float(np.trapz(Pxx[mask], f[mask]))

def spectral_entropy(Pxx: np.ndarray, eps: float = 1e-12) -> float:
    p = Pxx + eps
    p = p / np.sum(p)
    return float(-np.sum(p * np.log(p)))

def channel_features(x: np.ndarray, fs: int) -> dict:
    # Welch PSD
    f, Pxx = welch(x, fs=fs, nperseg=min(N_PER_SEG, len(x)))
    
    dom_f = float(f[np.argmax(Pxx)]) if len(Pxx) else 0.0
    
    bp_3_10 = bandpower(f, Pxx, 3.0, 10.0)
    se = spectral_entropy(Pxx)
    
    return {
        "mean": float(np.mean(x)),
        "std": float(np.std(x)),
        "rms": rms(x),
        "ptp": float(np.ptp(x)),
        "zcr": zero_crossing_rate(x),
        "dom_freq": dom_f,
        "bandpower_3_10": bp_3_10,
        "spec_entropy": se
    }


## Segment -> Feature-Vektor 

In [ ]:
def segment_to_features(sig_2ch: np.ndarray, fs: int) -> dict:
    feats = {}
    for ch in [0, 1]:
        x = sig_2ch[:, ch].astype(float)

        if BANDPASS_ON:
            x = bandpass_filter(x, fs=fs, low=BP_LOW, high=BP_HIGH, order=BP_ORDER)

        ch_feats = channel_features(x, fs=fs)
        for k, v in ch_feats.items():
            feats[f"{k}_ch{ch}"] = v

    x0 = sig_2ch[:, 0].astype(float)
    x1 = sig_2ch[:, 1].astype(float)
    if BANDPASS_ON:
        x0 = bandpass_filter(x0, fs=fs, low=BP_LOW, high=BP_HIGH, order=BP_ORDER)
        x1 = bandpass_filter(x1, fs=fs, low=BP_LOW, high=BP_HIGH, order=BP_ORDER)

    corr = np.corrcoef(x0, x1)[0, 1]
    feats["corr_ch0_ch1"] = float(corr) if np.isfinite(corr) else 0.0

    return feats


## Pro Record laden (alle Segmente featuren)

In [ ]:
def load_npz(npz_file: str):
    d = np.load(HARM_DIR / npz_file, allow_pickle=True)
    sig = d["signal"]
    fs = int(d["fs"])
    return sig, fs

features_rows = []


In [ ]:
for npz_file, grp in seg_index.groupby("npz_file"):
    sig, fs = load_npz(npz_file)
    assert fs == FS, f"{npz_file}: fs={fs} != {FS}"

    for _, row in grp.iterrows():
        seg = sig[int(row.start_idx):int(row.end_idx), :]  

        feats = segment_to_features(seg, fs=fs)

        out = {
            "source": row.source,
            "record": row.record,
            "start_s": float(row.start_s),
            "label": int(row.label),
            "npz_file": npz_file
        }
        out.update(feats)
        features_rows.append(out)

print("Done. Feature rows:", len(features_rows))


## Speichern

In [ ]:
features_df = pd.DataFrame(features_rows)
out_path = OUT_DIR / "features_5s_2p5s_clean.parquet"
features_df.to_parquet(out_path, index=False)
print("Saved:", out_path.resolve())
